In [30]:
from langchain_openai import ChatOpenAI
from dotenv import load_dotenv
from langgraph.graph import StateGraph,START,END
from langchain_core.runnables import RunnableBranch, RunnableParallel
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser,PydanticOutputParser
from typing import List,TypedDict,Dict,Literal
from pydantic import BaseModel,Field
load_dotenv(override=True)

model=ChatOpenAI()

class Sentiment(BaseModel): #schemadefination
    sentiment:Literal['positive','negative']=Field(description="Sentiment")
stroutputmodel=model.with_structured_output(Sentiment)

class Diagnosis(BaseModel):
    IssueType:Literal['UX','Software']=Field(description="IssueType")
    UserTone:Literal['Angry','Soft']=Field(description="UserTone")
stroutputmodel2=model.with_structured_output(Diagnosis)

class ReviewState(TypedDict):
    Review:str
    sentiment:Literal['positive','negative']
    Diagnosis:dict
    AIResponse:str

def RunDiagnosis(state:ReviewState):
    Prompt=f"Generate the IssueType and UserTone for the given {state['Review']}"
    result=stroutputmodel2.invoke(Prompt)
    return {'Diagnosis':result.model_dump()}
    

def findSentiment(state:ReviewState)->ReviewState:
    prompt=f"Generate the sentiment for the given {state['Review']}"
    result=stroutputmodel.invoke(prompt)
    return {'sentiment':result.sentiment}

def positiveAI(state:ReviewState):
    prompt=f"Generate the positive AI response for the review:{state['Review']}"
    result=model.invoke(prompt)
    return {'AIResponse':result.content}

def negativeAI(state:ReviewState):
    prompt=f"Generate the negative AI response for the review:{state['Review']}"
    result=model.invoke(prompt)
    return {'AIResponse':result.content}

def conditioncheck(state:ReviewState)->Literal['RunDiagnosis','positiveAI']:
    if state['sentiment']=='negative':
        return 'RunDiagnosis'
    else:
        return 'positiveAI'

graph=StateGraph(ReviewState)
graph.add_node('findSentiment',findSentiment)
graph.add_node('RunDiagnosis',RunDiagnosis)
graph.add_node('positiveAI',positiveAI)
graph.add_node('negativeAI',negativeAI)


graph.add_edge(START,'findSentiment')
graph.add_conditional_edges('findSentiment',conditioncheck)

graph.add_edge('RunDiagnosis','negativeAI')
graph.add_edge('negativeAI',END)
graph.add_edge('positiveAI',END)
graph.compile()

workflow=graph.compile()
initial_state={'Review':'I dont like this mobile'}
final_state=workflow.invoke(initial_state)
print(final_state)


d:\VSCode_AgenticAI\myenv\Lib\site-packages\langchain_openai\chat_models\base.py:2493: UserWarning: Cannot use method='json_schema' with model gpt-3.5-turbo since it doesn't support OpenAI's Structured Output API. You can see supported models here: https://platform.openai.com/docs/guides/structured-outputs#supported-models. To fix this warning, set `method='function_calling'. Overriding to method='function_calling'.
  warnings.warn(


{'Review': 'I dont like this mobile', 'sentiment': 'negative', 'Diagnosis': {'IssueType': 'UX', 'UserTone': 'Angry'}, 'AIResponse': "I'm sorry to hear that you are not satisfied with the mobile device. If you have any specific feedback or issues you would like to address, please let us know so we can try to assist you further."}
